# Traducción RU→EN para doblaje en Colab (Gemma local via Ollama)

**Antes de ejecutar:**
1. Runtime con GPU: `Entorno de ejecución → Cambiar tipo → T4 GPU`.
2. En tu Google Drive creá la carpeta `MyDrive/Doblaje/` con:
   - `RU/` → los pares `<video>.srt` + `<video>.json` (de faster-whisper-xxl)
   - `traducir_local.py` (de `Pipeline/` del repo)
3. Ejecutá las celdas en orden. La salida queda en `MyDrive/Doblaje/EN/`.

**Si Colab se desconecta**: volvé a ejecutar todas las celdas — el progreso vive en
`EN/<video>.parts.txt` en tu Drive y el script continúa donde quedó.

In [ ]:
# 1) Montar Drive y configurar rutas
from google.colab import drive
drive.mount('/content/drive')

BASE       = "/content/drive/MyDrive/Doblaje"   # editá si usás otra carpeta
CARPETA_RU = f"{BASE}/RU"
CARPETA_EN = f"{BASE}/EN"
SCRIPT     = f"{BASE}/traducir_local.py"
MODELO     = "hf.co/mradermacher/Huihui-gemma-4-E2B-it-abliterated-GGUF:Q4_K_M"

import os
os.makedirs(CARPETA_EN, exist_ok=True)
assert os.path.exists(SCRIPT), f"Falta {SCRIPT} — subí traducir_local.py a esa carpeta del Drive"
pares = sorted(f[:-4] for f in os.listdir(CARPETA_RU)
               if f.endswith('.srt') and os.path.exists(os.path.join(CARPETA_RU, f[:-4] + '.json')))
print(f"{len(pares)} pares srt+json encontrados:")
for p in pares: print("  -", p)

In [ ]:
# 2) Instalar Ollama, levantar el server y bajar el modelo (~3.4 GB, una vez por sesión)
import subprocess, time, urllib.request, os
!curl -fsSL https://ollama.com/install.sh | sh

env = dict(os.environ, OLLAMA_KEEP_ALIVE="4h")
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
        print("Ollama arriba"); break
    except Exception:
        time.sleep(1)

!ollama pull {MODELO}
!nvidia-smi -L

In [ ]:
# 3) Prueba del endpoint: una traducción suelta
import json, urllib.request
def chat(msg, max_tokens=80):
    body = json.dumps({"model": MODELO, "stream": False, "max_tokens": max_tokens,
                       "messages": [{"role": "user", "content": msg}]}).encode()
    req = urllib.request.Request("http://127.0.0.1:11434/v1/chat/completions",
                                 data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())["choices"][0]["message"]["content"]

print(chat("Translate to colloquial English, reply ONLY the translation: "
           "Ну вот, ребятки, сейчас покажу вам качер в работе."))

In [ ]:
# 4) Corrida de PRUEBA: 30 cues por video (medí tiempos y calidad antes de la completa)
!python "{SCRIPT}" --in "{CARPETA_RU}" --out "{CARPETA_EN}" --model "{MODELO}" --limit 30

In [ ]:
# 5) Corrida COMPLETA (reanudable: re-ejecutar esta celda continúa donde quedó)
!python "{SCRIPT}" --in "{CARPETA_RU}" --out "{CARPETA_EN}" --model "{MODELO}"

## Al terminar
- `EN/<video>-EN.srt` → el subtítulo final (timestamps originales intactos).
- `EN/<video>.flags.txt` → cues que el QA marcó para revisar (presupuesto, cirílico, fragmentos).
- `EN/<video>.parts.txt` → progreso crudo; se puede borrar cuando el srt final existe.

Para evaluar la calidad de Gemma: corré el video de streams (que ya tiene traducción de
referencia hecha por Claude en el repo), subí el `-EN.srt` resultante al repo y pedile a
Claude que lo compare cue por cue contra el gold (`Pipeline/RECETA-DOBLAJE.md`).

Si Ollama no soportara la arquitectura del GGUF (error en `ollama pull` o al generar),
alternativa en la misma celda 2: `pip install llama-cpp-python[server]` y levantar
`python -m llama_cpp.server --model <ruta.gguf> --n_gpu_layers -1 --port 11434` — el
script no cambia (mismo endpoint OpenAI-compatible).